> 在线讲义：[第 46 章 · CLIP 与 SigLIP：图和文字配不配，用点积打分](https://wikieden.github.io/beyond-transformer/lessons/46-clip-siglip.html) · 本书《超越 Transformer》

# 第 46 章 · CLIP 与 SigLIP：图和文字配不配，用点积打分

配套讲义：[在线版](https://wikieden.github.io/beyond-transformer/lessons/46-clip-siglip.html) · 仓库里 `beyond-transformer/notes/lessons/46-clip-siglip.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。只用 NumPy，不需要 GPU。练习题不在这里算，先在讲义里自己做。

In [ ]:
import numpy as np

np.set_printoptions(precision=4, suppress=True)  # 打印保留 4 位小数，不用科学计数法
print("numpy", np.__version__)

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 公式 | 写法 | 跑出来 |
|---|---|---|---|
| 第二步 分数表 | $S = I_e T_e^\top$ | `I @ T.T` | 对角线 0.96、0.96、1.0；鸟图 × 猫 = 0.936 |
| 第三步 按行 | 行 softmax + CE | `clip_loss(S, 10)["row"]` | 对角线概率 0.972、0.831、0.601，平均 loss ≈ 0.240 |
| 第三步 按列、总 loss | 对称 | `clip_loss(S, 10)` | 列 ≈ 0.303，总 ≈ 0.272 |
| 第四步 zero-shot | argmax | `zero_shot([0.96, 0.28], T, 10)` | 狗 0.873、猫 0.009、鸟 0.118 |
| 第五步 SigLIP | sigmoid 损失 | `siglip_loss(S, 10, -10)` | 对角线每格 0.913、0.913、0.693，总 ≈ 1.07 |

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

In [ ]:
I = np.array([[1, 0], [0, 1], [0.6, 0.8]])  # 狗图、猫图、鸟图（编的，已归一化）
T = np.array([[0.96, 0.28], [0.28, 0.96], [0.6, 0.8]])  # "一只狗"、"一只猫"、"一只鸟"

**函数 `softmax`**：沿 axis 做 softmax（先减最大值，第 1 章）。

In [ ]:
def softmax(x, axis):
    """沿 axis 做 softmax（先减最大值，第 1 章）。"""
    e = np.exp(x - x.max(axis=axis, keepdims=True))
    return e / e.sum(axis=axis, keepdims=True)

**函数 `sigmoid`**：σ(x) = 1 / (1 + e^(−x))。

In [ ]:
def sigmoid(x):
    """σ(x) = 1 / (1 + e^(−x))。"""
    return 1 / (1 + np.exp(-x))

**函数 `clip_loss`**：CLIP 对称交叉熵：按行（图找文）、按列（文找图）各算对角线的 −log，再平均。

In [ ]:
def clip_loss(S, scale):
    """CLIP 对称交叉熵：按行（图找文）、按列（文找图）各算对角线的 −log，再平均。"""
    logits = scale * S
    row = -np.log(np.diag(softmax(logits, axis=1)))
    col = -np.log(np.diag(softmax(logits, axis=0)))
    return {"row_p": np.diag(softmax(logits, axis=1)), "row": row.mean(), "col": col.mean(),
            "loss": (row.mean() + col.mean()) / 2}

**函数 `zero_shot`**：zero-shot：图片向量和每个类名句子的向量比余弦，放大后 softmax。

In [ ]:
def zero_shot(img, texts, scale):
    """zero-shot：图片向量和每个类名句子的向量比余弦，放大后 softmax。"""
    return softmax(scale * (texts @ np.asarray(img)), axis=0)

**函数 `siglip_loss`**：SigLIP：每格 −log σ(z · (t·cos + b))，z 对角线 +1、其余 −1；总和除以 |B|。

In [ ]:
def siglip_loss(S, t, b):
    """SigLIP：每格 −log σ(z · (t·cos + b))，z 对角线 +1、其余 −1；总和除以 |B|。"""
    n = len(S)
    z = 2 * np.eye(n) - 1
    cell = -np.log(sigmoid(z * (t * S + b)))
    return {"logits": t * S + b, "cell": cell, "loss": cell.sum() / n}

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第二步 分数表 S = I · Tᵀ")
S = I @ T.T
print(S)
print("放大 10 倍：")
print(10 * S)

In [ ]:
print("\n第三步 对称交叉熵")
r = clip_loss(S, 10)
print("按行 softmax 的对角线：", r["row_p"])
print(f"按行 {r['row']:.3f}，按列 {r['col']:.3f}，总 {r['loss']:.3f}")

In [ ]:
print("\n第四步 zero-shot：新照片 [0.96, 0.28]")
print("狗、猫、鸟：", zero_shot([0.96, 0.28], T, 10))

In [ ]:
print("\n第五步 SigLIP：t = 10，b = −10")
s = siglip_loss(S, 10, -10)
print(s["logits"])
print(s["cell"])
print(f"loss = {s['loss']:.3f}")

## 3. 代码里才看得到的

- CLIP 伪代码里 `axis=0` 是"按列做 softmax"，也就是"文字找图"那一边；和讲义里的行 / 列一一对应时别看反。
- SigLIP 的总和除的是 |B|（对数），不是 |B|² 个格子，所以数值看起来比 CLIP 大。

## 4. 自己改一改

- 把放大倍数从 10 改成 1，看按行 softmax 是不是变得几乎均匀，loss 变大。
- 把 SigLIP 的 b 改成 0，看对角线和非对角线格子的 loss 各怎么变。

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 图走图的编码器，文走文的编码器；投影归一化，点积就是余弦。
> 一批 N 对打 N 方格，对角线上是正解；温度放大分得开。
> 按行选文、按列选图，交叉熵两边各一半；一批越大反例越多。
> 类名套成一句话，最像的就是答案。
> SigLIP 每格判一判，sigmoid 加偏置，不要整行分母。